# Credit Scoring Data Analysis
## Analyzing Borrowers' Risk of Defaulting

This project analyzes borrower information for a bank's loan division. The goal is to explore whether customer characteristics such as marital status, number of children, income level, and loan purpose are associated with loan repayment behavior.

The analysis follows a complete data-analysis workflow: data inspection, preprocessing, exploratory data analysis, visualization, feature categorization, and answering business questions relevant to credit scoring.

## Objectives
- Understand the structure and quality of the borrower dataset.
- Identify and handle missing values and duplicated records.
- Inspect inconsistent and unusual values.
- Prepare categorical and numerical variables for analysis.
- Explore borrower characteristics and loan repayment behavior.
- Answer key credit-risk questions using descriptive analysis and visualizations.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns', None)
sns.set_context('notebook')

## 2. Load the Dataset
The dataset contains borrower demographic, employment, income, family, debt, and loan-purpose information.

In [ ]:
df = pd.read_csv('credit_scoring_eng.csv')
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.info()

### Dataset Columns
- `children`: number of children in the family
- `days_employed`: total employment experience in days
- `dob_years`: customer age in years
- `education`: education level
- `education_id`: encoded education level
- `family_status`: marital/family status
- `family_status_id`: encoded family status
- `gender`: customer gender
- `income_type`: type/source of employment income
- `debt`: whether the customer has previously failed to repay a loan on time
- `total_income`: monthly income
- `purpose`: stated purpose of the loan

## 3. Initial Data Exploration

In [ ]:
df.describe(include='all').T

In [ ]:
df.nunique().sort_values()

The initial inspection is useful for identifying missing values, incorrect data types, duplicated records, inconsistent category labels, and unusual numerical values before analysis.

## 4. Missing Values

In [ ]:
missing_count = df.isna().sum()
missing_percent = (missing_count / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Values': missing_count, 'Percentage': missing_percent})
missing_summary[missing_summary['Missing Values'] > 0]

In [ ]:
df[df['days_employed'].isna() | df['total_income'].isna()].head()

The original analysis identifies missing observations in `days_employed` and `total_income`. Because both variables are important to the analysis, we inspect their missingness before preprocessing.

In [ ]:
df.groupby('gender')[['days_employed', 'total_income']].apply(lambda x: x.isna().sum())

### Missing-Value Treatment
Following the project workflow, rows containing missing values are removed before the remaining preprocessing and analysis steps.

In [ ]:
df_clean = df.dropna().copy()
print('Original shape:', df.shape)
print('Shape after removing missing rows:', df_clean.shape)
print('Remaining missing values:', df_clean.isna().sum().sum())

## 5. Inspect Unique Values and Data Quality

In [ ]:
for column in df_clean.columns:
    if df_clean[column].nunique() <= 30:
        print(f'\n{column}:')
        print(df_clean[column].value_counts(dropna=False))

### Number of Children
Inspect the `children` column for impossible or unusual values before using it in analysis.

In [ ]:
df_clean['children'].value_counts().sort_index()

In [ ]:
df_clean.loc[df_clean['children'] == -1, 'children'] = 0
df_clean['children'].value_counts().sort_index()

### Employment Days
Negative employment-day values are converted to their absolute magnitude for easier interpretation.

In [ ]:
df_clean['days_employed'] = df_clean['days_employed'].abs()
df_clean['days_employed'].describe()

### Standardize Education Labels
The same education category appears with different capitalization. Converting text to lowercase makes the categories consistent.

In [ ]:
df_clean['education'] = df_clean['education'].str.lower().str.strip()
df_clean['education'].value_counts()

## 6. Data Type Conversion

In [ ]:
df_clean['days_employed'] = df_clean['days_employed'].astype(int)
df_clean['total_income'] = df_clean['total_income'].astype(int)
df_clean.dtypes

## 7. Duplicate Records

In [ ]:
print('Duplicated rows:', df_clean.duplicated().sum())
df_clean[df_clean.duplicated()].head()

In [ ]:
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print('Shape after duplicate removal:', df_clean.shape)

## 8. Exploratory Data Analysis

### 8.1 Numerical Variables

In [ ]:
numerical_columns = ['children', 'days_employed', 'dob_years', 'total_income']
df_clean[numerical_columns].describe().T

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df_clean, x=column, kde=True)
    plt.title(f'Distribution of {column}')
    plt.tight_layout()
    plt.show()

### 8.2 Outlier Inspection

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(8, 3))
    sns.boxplot(data=df_clean, x=column)
    plt.title(f'Boxplot of {column}')
    plt.tight_layout()
    plt.show()

### 8.3 Categorical Variables

In [ ]:
categorical_columns = ['education', 'family_status', 'gender', 'income_type', 'debt']
for column in categorical_columns:
    print(f'\n--- {column} ---')
    print(df_clean[column].value_counts())

In [ ]:
for column in ['education', 'family_status', 'gender', 'income_type']:
    plt.figure(figsize=(10, 5))
    order = df_clean[column].value_counts().index
    sns.countplot(data=df_clean, y=column, order=order)
    plt.title(f'Distribution of {column}')
    plt.tight_layout()
    plt.show()

## 9. Correlation Analysis
The correlation matrix provides an initial view of linear relationships among numerical variables. Correlation alone does not establish causation.

In [ ]:
numeric_df = df_clean.select_dtypes(include=np.number)
correlation = numeric_df.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 10. Categorizing Loan Purpose
The free-text `purpose` column contains several phrases that refer to the same broad loan purpose. We group them into consistent categories for analysis.

In [ ]:
def categorize_purpose(text):
    text = str(text).lower()
    if any(word in text for word in ['house', 'housing', 'property', 'real estate']):
        return 'Housing'
    if any(word in text for word in ['car', 'vehicle']):
        return 'Vehicle'
    if any(word in text for word in ['education', 'university', 'educated']):
        return 'Education'
    if 'wedding' in text:
        return 'Wedding'
    return 'Other'

df_clean['purpose_category'] = df_clean['purpose'].apply(categorize_purpose)
df_clean['purpose_category'].value_counts()

## 11. Business Question 1: Is There a Relationship Between Having Children and Repaying a Loan on Time?
The `debt` variable is used as the repayment-risk indicator. We calculate the mean debt rate for each number of children.

In [ ]:
children_debt = df_clean.groupby('children').agg(
    customers=('debt', 'size'),
    defaults=('debt', 'sum'),
    default_rate=('debt', 'mean')
).reset_index()
children_debt['default_rate_percent'] = (children_debt['default_rate'] * 100).round(2)
children_debt

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=children_debt, x='children', y='default_rate_percent')
plt.ylabel('Default Rate (%)')
plt.title('Loan Default Rate by Number of Children')
plt.tight_layout()
plt.show()

### Interpretation
Compare the default percentages together with the number of customers in each group. Groups with very few observations should be interpreted cautiously.

## 12. Business Question 2: Is There a Relationship Between Marital Status and Repaying a Loan on Time?

In [ ]:
family_debt = df_clean.groupby('family_status').agg(
    customers=('debt', 'size'),
    defaults=('debt', 'sum'),
    default_rate=('debt', 'mean')
).sort_values('default_rate', ascending=False)
family_debt['default_rate_percent'] = (family_debt['default_rate'] * 100).round(2)
family_debt

In [ ]:
plt.figure(figsize=(10, 5))
family_plot = family_debt.reset_index()
sns.barplot(data=family_plot, y='family_status', x='default_rate_percent')
plt.xlabel('Default Rate (%)')
plt.ylabel('Family Status')
plt.title('Loan Default Rate by Family Status')
plt.tight_layout()
plt.show()

## 13. Children and Family Status Together

In [ ]:
children_family_pivot = pd.pivot_table(
    df_clean,
    index='family_status',
    columns='children',
    values='debt',
    aggfunc='mean'
)
children_family_pivot

## 14. Education, Family Status, and Debt

In [ ]:
education_family_pivot = pd.pivot_table(
    df_clean,
    index='education',
    columns='family_status',
    values='debt',
    aggfunc='mean'
)
education_family_pivot

## 15. Business Question 3: Is There a Relationship Between Income Level and Repaying a Loan on Time?
Income is continuous, so we create income groups using quantiles and compare default rates across them.

In [ ]:
df_clean['income_group'] = pd.qcut(
    df_clean['total_income'],
    q=4,
    labels=['Low', 'Lower-Middle', 'Upper-Middle', 'High'],
    duplicates='drop'
)
income_debt = df_clean.groupby('income_group', observed=False).agg(
    customers=('debt', 'size'),
    average_income=('total_income', 'mean'),
    default_rate=('debt', 'mean')
)
income_debt['default_rate_percent'] = (income_debt['default_rate'] * 100).round(2)
income_debt

In [ ]:
plt.figure(figsize=(8, 5))
income_plot = income_debt.reset_index()
sns.barplot(data=income_plot, x='income_group', y='default_rate_percent')
plt.xlabel('Income Group')
plt.ylabel('Default Rate (%)')
plt.title('Loan Default Rate by Income Group')
plt.tight_layout()
plt.show()

## 16. Income by Gender and Income Type

In [ ]:
income_gender = df_clean.groupby('gender')['total_income'].agg(['count', 'mean', 'median'])
income_gender

In [ ]:
income_type_summary = df_clean.groupby('income_type').agg(
    customers=('total_income', 'size'),
    mean_income=('total_income', 'mean'),
    median_income=('total_income', 'median'),
    default_rate=('debt', 'mean')
).sort_values('mean_income', ascending=False)
income_type_summary

## 17. Business Question 4: How Do Different Loan Purposes Affect Repayment?

In [ ]:
purpose_debt = df_clean.groupby('purpose_category').agg(
    customers=('debt', 'size'),
    defaults=('debt', 'sum'),
    default_rate=('debt', 'mean')
).sort_values('default_rate', ascending=False)
purpose_debt['default_rate_percent'] = (purpose_debt['default_rate'] * 100).round(2)
purpose_debt

In [ ]:
plt.figure(figsize=(9, 5))
purpose_plot = purpose_debt.reset_index()
sns.barplot(data=purpose_plot, x='purpose_category', y='default_rate_percent')
plt.xlabel('Loan Purpose')
plt.ylabel('Default Rate (%)')
plt.title('Loan Default Rate by Purpose')
plt.tight_layout()
plt.show()

## 18. Additional Cross-Analysis

In [ ]:
pd.crosstab(df_clean['family_status'], df_clean['debt'], normalize='index').round(3)

In [ ]:
pd.crosstab(df_clean['education'], df_clean['debt'], normalize='index').round(3)

In [ ]:
pd.crosstab(df_clean['income_type'], df_clean['debt'], normalize='index').round(3)

## 19. Final Data Quality Check

In [ ]:
print('Final shape:', df_clean.shape)
print('Missing values:', df_clean.isna().sum().sum())
print('Duplicate rows:', df_clean.duplicated().sum())
df_clean.head()

## 20. General Conclusion
This project demonstrates a complete preprocessing and exploratory-analysis workflow for borrower credit data. The analysis investigates how children, family status, income, education, employment type, and loan purpose relate to historical repayment behavior.

The results are descriptive associations in this dataset and should not be interpreted as proof that any demographic characteristic causes loan default. For a production credit-scoring system, additional validation, fairness assessment, feature engineering, and predictive-model evaluation would be required.

## Skills Demonstrated
- Python programming
- pandas and NumPy
- Data cleaning and preprocessing
- Missing-value analysis
- Duplicate and inconsistent-value handling
- Exploratory Data Analysis (EDA)
- Data visualization with Matplotlib and Seaborn
- GroupBy, pivot tables, and cross-tabulation
- Feature categorization
- Business-question analysis
- Credit-risk data interpretation